# Setup
Setup dell'ambiente per far funzionare selenium in colab
- installazione di chromium e chromedriver
- installazione di selenium

In [ ]:
# 1. Rimuovi eventuali residui chromium (non necessario ma pulisce confusione)
!apt-get purge -y chromium-browser chromium-chromedriver -qq 2>/dev/null

# 2. Installa Google Chrome vero (non è uno snap, è un .deb classico)
!wget -q -O /tmp/chrome.deb https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt-get install -y /tmp/chrome.deb -qq

# 3. Verifica che sia installato correttamente
!google-chrome --version

Selecting previously unselected package at-spi2-common.
(Reading database ... 122809 files and directories currently installed.)
Preparing to unpack .../0-at-spi2-common_2.52.0-1build1_all.deb ...
Unpacking at-spi2-common (2.52.0-1build1) ...
Selecting previously unselected package libatk1.0-0t64:amd64.
Preparing to unpack .../1-libatk1.0-0t64_2.52.0-1build1_amd64.deb ...
Unpacking libatk1.0-0t64:amd64 (2.52.0-1build1) ...
Selecting previously unselected package libatspi2.0-0t64:amd64.
Preparing to unpack .../2-libatspi2.0-0t64_2.52.0-1build1_amd64.deb ...
Unpacking libatspi2.0-0t64:amd64 (2.52.0-1build1) ...
Selecting previously unselected package libatk-bridge2.0-0t64:amd64.
Preparing to unpack .../3-libatk-bridge2.0-0t64_2.52.0-1build1_amd64.deb ...
Unpacking libatk-bridge2.0-0t64:amd64 (2.52.0-1build1) ...
Selecting previously unselected package libxcomposite1:amd64.
Preparing to unpack .../4-libxcomposite1_1%3a0.4.5-1build3_amd64.deb ...
Unpacking libxcomposite1:amd64 (1:0.4.5-1bu

In [ ]:
!pip install selenium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.8/511.8 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.7/135.7 kB 7.9 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0


## Copia del driver di selenium



In [ ]:
!cp /usr/lib/chromium-browser/chromedriver /usr/bin

cp: cannot stat '/usr/lib/chromium-browser/chromedriver': No such file or directory


In [ ]:
from selenium.webdriver.common.by import By
from selenium.common import NoSuchElementException, ElementNotInteractableException, TimeoutException

In [ ]:
SITE_CONFIG = {
    'bollettino_puglia': {
        'selectors':{
            'url': 'https://protezionecivile.regione.puglia.it/bollettino-di-criticit%C3%A0',
            'list_block': (By.CLASS_NAME, "lista-servizi-ricerca"),
            'block_name': (By.CLASS_NAME, 'card-servizio'),
            'cookie_button': (By.CLASS_NAME, 'c-bn'),
            'close_pop_up': (By.ID, 'close-pop-up-button'),
           # 'page_element': (By.CLASS_NAME, 'toolbar-number'),
            #'href_link': (By.CLASS_NAME, 'product-item-link'),


        },
        'wait_time': 2,
        'pagination_param': 'p',
        'query_cat':'q'
    },
    'bollettino_incendi': {
        # stessa struttura della pagina di criticità (stesso modulo "Cerca tra i Bollettini")
        'selectors':{
            'url': 'https://protezionecivile.regione.puglia.it/bollettini-incendi-boschivi',
            'list_block': (By.CLASS_NAME, "lista-servizi-ricerca"),
            'block_name': (By.CLASS_NAME, 'card-servizio'),
            'cookie_button': (By.CLASS_NAME, 'c-bn'),
            'close_pop_up': (By.ID, 'close-pop-up-button'),
        },
        'wait_time': 2,
        'pagination_param': 'p',
        'query_cat':'q'
    }
}

# Init configurazione e scraper


In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options

import time
import os


In [ ]:
def get_config(site_key):
    """Recupera la configurazione per un sito specifico"""
    return SITE_CONFIG[site_key]

nel codice, occorre specificare esplicitamente il `binary_location` di Chrome

In [ ]:
!pip install -U selenium webdriver-manager --quiet

In [ ]:

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

def init_driver():
    pathname = '/content/data'
    options = Options()
    options.binary_location = '/usr/bin/google-chrome'
    options.add_argument('--headless=new')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('--disable-gpu')
    options.add_argument('--window-size=1920,1080')
    options.add_experimental_option("prefs", {
        "download.default_directory": pathname,
        "download.prompt_for_download": False,
        "download.directory_upgrade": True,
        "safebrowsing.enabled": True
    })

    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=options)
    return driver

driver = init_driver()
print(driver.title)  # test rapido
driver.quit()

In [ ]:
from selenium.common import NoSuchElementException, TimeoutException, ElementNotInteractableException


def handle_popup(driver):
    try:
        # Click the checkboxes
        policy = driver.find_element(By.XPATH, "//span[@id='spanCB1']")
        policy.click()

        policy2 = driver.find_element(By.XPATH, "//span[@id='spanCB2']")
        policy2.click()

        # Click the button to show more options
        policy3 = driver.find_element(By.XPATH, "//button[@type='button']")
        policy3.click()
        #close_pop_up = driver.find_element(*popup_selector)
        #driver.execute_script("arguments[0].click();", close_pop_up)
        print("popup button closed: OK")
    except (NoSuchElementException, TimeoutException, ElementNotInteractableException):
        print("Unable to close popup button: KO")
    except Exception as e:
        print(f"An unknown error occurred: KO ->  {e}")

In [ ]:
!apt-get install -y locales
!locale-gen it_IT.UTF-8
!update-locale LANG=it_IT.UTF-8

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
locales is already the newest version (2.39-0ubuntu8.9).
0 upgraded, 0 newly installed, 0 to remove and 0 not upgraded.
Generating locales (this might take a while)...
  it_IT.UTF-8... done
Generation complete.


In [ ]:
from datetime import datetime

MESI_IT = ["gennaio", "febbraio", "marzo", "aprile", "maggio", "giugno", "luglio",
           "agosto", "settembre", "ottobre", "novembre", "dicembre"]

def data_oggi_ita(oggi=None):
    oggi = oggi or datetime.now()
    return f"{oggi.day} {MESI_IT[oggi.month - 1]} {oggi.year}"


# Download


In [ ]:
import requests

def scarica_pdf(href, cartella='/content/data', nome_file=None):
    os.makedirs(cartella, exist_ok=True)

    if nome_file is None:
        oggi_str = data_oggi_ita().replace(" ", "_")
        nome_file = f"bollettino_{oggi_str}.pdf"

    percorso_completo = os.path.join(cartella, nome_file)

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
    }

    r = requests.get(href, headers=headers, timeout=30)
    r.raise_for_status()  # solleva errore se status != 200
    if not r.content.startswith(b'%PDF'):
        raise ValueError(f"Il link non restituisce un PDF (Content-Type: {r.headers.get('Content-Type')}): {href}")

    with open(percorso_completo, 'wb') as f:
        f.write(r.content)

    print(f"PDF salvato in: {percorso_completo}")
    return percorso_completo

In [ ]:
import pandas as pd
from urllib.parse import urlparse, unquote


def nome_file_da_url(href, fallback):
  """Nome originale del PDF preso dal link (es. 20260925_106_BollettinoIncendi_784.pdf).
  Nei link del portale il nome è un segmento intermedio: /documents/.../NOME.pdf/<uuid>?t=..."""
  segmenti = [unquote(s) for s in urlparse(href).path.split('/') if s.lower().endswith('.pdf')]
  return segmenti[-1] if segmenti else fallback


def crawl_from_last_date(site_key, cartella='/content/data'):
  """Scarica tutti i bollettini pubblicati oggi sulla pagina di `site_key`. Restituisce i percorsi salvati."""
  config = get_config(site_key)
  url = config['selectors']['url']
  today = data_oggi_ita()
  scaricati = []

  driver = init_driver()
  try:
    print(f'url bollettino: {url}')
    driver.get(url)
    # la lista dei risultati viene caricata dopo la pagina: aspetta che compaia
    container = WebDriverWait(driver, 15).until(
        EC.presence_of_element_located(config['selectors']['list_block']))
    block_list = container.find_elements(By.XPATH, "./*")  # tutti i figli diretti
    print("Iterating through block of length:", len(block_list))

    for block in block_list:
      try:
        data_text = block.find_element(By.CLASS_NAME, "data-news").text.strip()
      except NoSuchElementException:
        continue  # blocco senza data (es. separatori)
      print(f"Confronto: '{data_text}' vs '{today}'")
      if data_text != today:
        continue
      href = block.find_element(By.TAG_NAME, "a").get_attribute("href")
      nome = nome_file_da_url(href, fallback=f"{site_key}_{today.replace(' ', '_')}.pdf")
      print(f"Trovata card di oggi: {block.text!r}\n  Link: {href}")
      scaricati.append(scarica_pdf(href, cartella, nome))
      time.sleep(config['wait_time'])
  except TimeoutException:
    print(f"Lista dei bollettini non trovata su {url}: la pagina potrebbe avere una struttura diversa")
  finally:
    driver.quit()

  if not scaricati:
    print('Nessuna card di oggi trovata')
  return scaricati

In [ ]:
crawl_from_last_date('bollettino_puglia')

url bollettino: https://protezionecivile.regione.puglia.it/bollettino-di-criticit%C3%A0
Iterating through block of length: 10
Confronto: '25 settembre 2026' vs '25 settembre 2026'
Trovata card di oggi: '25 settembre 2026\nBollettino di Criticità Regionale del 25/09/2026'
  Link: https://protezionecivile.regione.puglia.it/documents/3171874/3182007/20260925_BCR_269.pdf/a1148b9a-5428-ca86-44c9-8040cf61b665?t=1790330920430
PDF salvato in: /content/data/20260925_BCR_269.pdf
Confronto: '24 settembre 2026' vs '25 settembre 2026'
Confronto: '23 settembre 2026' vs '25 settembre 2026'
Confronto: '22 settembre 2026' vs '25 settembre 2026'
Confronto: '21 settembre 2026' vs '25 settembre 2026'
Confronto: '20 settembre 2026' vs '25 settembre 2026'
Confronto: '19 settembre 2026' vs '25 settembre 2026'
Confronto: '18 settembre 2026' vs '25 settembre 2026'
Confronto: '17 settembre 2026' vs '25 settembre 2026'
Confronto: '16 settembre 2026' vs '25 settembre 2026'


['/content/data/20260925_BCR_269.pdf']

## Bollettino incendi boschivi
Scarica il bollettino regionale di previsione incendi di oggi. Il file mantiene il nome originale (es. `20260925_106_BollettinoIncendi_784.pdf`), così non si sovrappone al bollettino di criticità dello stesso giorno.

In [ ]:
pdf_incendi = crawl_from_last_date('bollettino_incendi')

url bollettino: https://protezionecivile.regione.puglia.it/bollettini-incendi-boschivi
Iterating through block of length: 10
Confronto: '25 settembre 2026' vs '25 settembre 2026'
Trovata card di oggi: '25 settembre 2026\n25/09/2026 - Ore 10.06 - Bollettino Regionale Incendi Boschivi'
  Link: https://protezionecivile.regione.puglia.it/documents/3171874/3219213/20260925_106_BollettinoIncendi_784.pdf/dc58dc00-5869-0ece-2f0b-12b5b24394ac?t=1790324652677
PDF salvato in: /content/data/20260925_106_BollettinoIncendi_784.pdf
Confronto: '24 settembre 2026' vs '25 settembre 2026'
Confronto: '23 settembre 2026' vs '25 settembre 2026'
Confronto: '22 settembre 2026' vs '25 settembre 2026'
Confronto: '21 settembre 2026' vs '25 settembre 2026'
Confronto: '20 settembre 2026' vs '25 settembre 2026'
Confronto: '19 settembre 2026' vs '25 settembre 2026'
Confronto: '18 settembre 2026' vs '25 settembre 2026'
Confronto: '17 settembre 2026' vs '25 settembre 2026'
Confronto: '16 settembre 2026' vs '25 settemb

# Estrazione rischio incendi → `rischio.csv`
Legge le tabelle "Zona Omogenea AIB / Livello Pericolosità" dei **Bollettini regionali di previsione incendi** (una tabella per ciascun giorno: 24, 48 e 72 ore) e produce una riga per data × zona.

Come funziona:
- il PDF scrive ogni livello più volte sovrapposto (finto grassetto): `dedupe_chars()` elimina i duplicati;
- ogni livello viene assegnato alla zona **in base alla posizione x** della colonna nell'intestazione, non all'ordine del testo;
- se una tabella non ha esattamente 16 livelli, la funzione si ferma con un errore invece di scrivere dati sbagliati.

In [ ]:
!pip install pdfplumber --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 67.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 107.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 102.2 MB/s eta 0:00:00


In [ ]:
import re, pdfplumber

ZONE = ["BAT","BA_01","BA_02","BA_03","BR_01","BR_02","FG_01","FG_02",
        "FG_03","FG_04","LE_01","LE_02","LE_03","TA_01","TA_02","TA_03"]
LIVELLI = ["BASSO", "MEDIO", "MODERATO", "ELEVATO", "ESTREMO"]
RE_LIV = re.compile(r"BAS{1,2}O|MEDIO|MODERATO|ELEVATO|ESTREMO")  # BAS{1,2}O: il dedup può fondere la doppia S
RE_DATA = re.compile(r"\d{2}/\d{2}/\d{4}")
iso = lambda d: f"{d[6:]}-{d[3:5]}-{d[:2]}" if d else None  # gg/mm/aaaa -> aaaa-mm-gg
norm = lambda t: "BASSO" if t.startswith("BAS") else t

def estrai_rischio(pdf_path):
    righe = []
    with pdfplumber.open(pdf_path) as pdf:
        testo_p1 = pdf.pages[0].extract_text() or ""
        m = re.search(r"Bollettino previsionale del (\d{2}/\d{2}/\d{4})", testo_p1)
        data_emissione = m.group(1) if m else None
        for page in pdf.pages:
            page = page.dedupe_chars()
            words = page.extract_words()
            for bat in [w for w in words if w["text"] == "BAT"]:
                y = bat["top"]
                # intestazione: posizione x di ogni zona sulla stessa riga di BAT
                header = {w["text"]: (w["x0"] + w["x1"]) / 2 for w in words
                          if w["text"] in ZONE and abs(w["top"] - y) < 2}
                if len(header) != 16:
                    continue  # etichetta "BAT" sulla mappa, non intestazione di tabella
                # data della tabella: a sinistra, subito sopra l'intestazione
                date = [w for w in words if RE_DATA.fullmatch(w["text"])
                        and w["x0"] < bat["x0"] and 0 < y - w["top"] < 40]
                if not date:
                    raise ValueError(f"{pdf_path} p.{page.page_number}: data tabella non trovata")
                data_prev = min(date, key=lambda w: y - w["top"])["text"]
                # riga dei livelli: caratteri sotto l'intestazione, a destra dell'etichetta
                chars = sorted((c for c in page.chars
                                if y + 10 < c["top"] < y + 40 and c["x0"] > bat["x0"] - 10),
                               key=lambda c: c["x0"])
                s = "".join(c["text"] for c in chars)
                assegnati = {}
                for tok in RE_LIV.finditer(s):
                    cs = chars[tok.start():tok.end()]
                    xc = (cs[0]["x0"] + cs[-1]["x1"]) / 2
                    zona = min(header, key=lambda z: abs(header[z] - xc))
                    if zona in assegnati:
                        raise ValueError(f"{pdf_path} {data_prev}: due livelli per {zona}")
                    assegnati[zona] = tok.group()
                if len(assegnati) != 16:
                    raise ValueError(f"{pdf_path} {data_prev}: {len(assegnati)} livelli su 16 ({s!r})")
                righe += [{"data_emissione": iso(data_emissione), "data_previsione": iso(data_prev),
                           "zona": z, "livello": norm(assegnati[z]),
                           "livello_num": LIVELLI.index(norm(assegnati[z])) + 1} for z in ZONE]
    if not righe:
        raise ValueError(f"{pdf_path}: nessuna tabella dei livelli trovata")
    return righe

## Controllo incrociato con i colori delle celle
Confronta il livello letto dal testo con il colore di sfondo della cella. I colori di BASSO, MEDIO e MODERATO sono stati verificati sul bollettino del 25/09/2026. Quelli di ELEVATO ed ESTREMO **non li ho ancora visti in un PDF reale**: se compaiono, vengono segnalati come "colore non mappato" da controllare a mano.

In [ ]:
COLORI = {(0.0, 1.0, 0.0): "BASSO", (1.0, 1.0, 0.0): "MEDIO", (1.0, 0.75, 0.0): "MODERATO"}

def controlla_colori(pdf_path, righe):
    """Restituisce le discrepanze tra testo e colore della cella (lista vuota = tutto ok)."""
    attesi = {(r["data_previsione"], r["zona"]): r["livello"] for r in righe}
    problemi = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            words = page.extract_words()
            for bat in [w for w in words if w["text"] == "BAT"]:
                y = bat["top"]
                header = {w["text"]: (w["x0"] + w["x1"]) / 2 for w in words
                          if w["text"] in ZONE and abs(w["top"] - y) < 2}
                if len(header) != 16:
                    continue
                data_prev = min((w for w in words if RE_DATA.fullmatch(w["text"])
                                 and w["x0"] < bat["x0"] and 0 < y - w["top"] < 40),
                                key=lambda w: y - w["top"])["text"]
                data_prev = iso(data_prev)
                celle = [r for r in page.rects if r["top"] < y + 30 < r["bottom"]
                         and r["x0"] > bat["x0"] - 10 and r["width"] < 40
                         and r["non_stroking_color"] and len(r["non_stroking_color"]) == 3
                         and tuple(round(v, 2) for v in r["non_stroking_color"]) != (0.83, 0.83, 0.83)]  # grigio = sfondo intestazione
                for z, xc in header.items():
                    cella = [r for r in celle if r["x0"] <= xc <= r["x1"]]
                    if not cella:
                        problemi.append((data_prev, z, "cella colorata non trovata"))
                        continue
                    col = tuple(round(v, 2) for v in cella[0]["non_stroking_color"])
                    liv_colore = COLORI.get(col, f"colore non mappato {col}")
                    if liv_colore != attesi.get((data_prev, z)):
                        problemi.append((data_prev, z, f"testo={attesi.get((data_prev, z))} colore={liv_colore}"))
    return problemi

## Elaborazione di tutti i PDF in `/content/data` e salvataggio
I file che non contengono la tabella (per esempio i Bollettini di Criticità) vengono saltati con un messaggio. Il CSV viene aggiornato senza duplicare le righe già presenti.

In [ ]:
import glob

CARTELLA = "/content/data"
CSV_OUT = os.path.join(CARTELLA, "rischio.csv")

nuove = []
for pdf_path in sorted(glob.glob(os.path.join(CARTELLA, "*.pdf"))):
    try:
        righe = estrai_rischio(pdf_path)
    except Exception as e:  # PDF senza tabella, corrotto o con layout cambiato
        print(f"SALTATO {os.path.basename(pdf_path)}: {e}")
        continue
    problemi = controlla_colori(pdf_path, righe)
    if problemi:
        print(f"ATTENZIONE {os.path.basename(pdf_path)}: {problemi}")
    for r in righe:
        r["file"] = os.path.basename(pdf_path)
    nuove += righe
    print(f"OK {os.path.basename(pdf_path)}: {len(righe)} righe")

df = pd.DataFrame(nuove)
if os.path.exists(CSV_OUT):
    df = pd.concat([pd.read_csv(CSV_OUT, dtype=str), df.astype(str)])
chiave = ["data_emissione", "data_previsione", "zona"]
if not df.empty:
    df = df.drop_duplicates(subset=chiave, keep="last").sort_values(chiave)
    df.to_csv(CSV_OUT, index=False)
    print(f"Salvato {CSV_OUT}: {len(df)} righe")
df.head(20)

OK 20260925_106_BollettinoIncendi_784.pdf: 48 righe
SALTATO 20260925_BCR_269.pdf: /content/data/20260925_BCR_269.pdf: nessuna tabella dei livelli trovata
Salvato /content/data/rischio.csv: 48 righe


,data_emissione,data_previsione,zona,livello,livello_num,file
0,2026-09-25,2026-09-25,BAT,MEDIO,2,20260925_106_BollettinoIncendi_784.pdf
1,2026-09-25,2026-09-25,BA_01,MEDIO,2,20260925_106_BollettinoIncendi_784.pdf
2,2026-09-25,2026-09-25,BA_02,MEDIO,2,20260925_106_BollettinoIncendi_784.pdf
3,2026-09-25,2026-09-25,BA_03,MEDIO,2,20260925_106_BollettinoIncendi_784.pdf
4,2026-09-25,2026-09-25,BR_01,MODERATO,3,20260925_106_BollettinoIncendi_784.pdf
5,2026-09-25,2026-09-25,BR_02,MEDIO,2,20260925_106_BollettinoIncendi_784.pdf
6,2026-09-25,2026-09-25,FG_01,MODERATO,3,20260925_106_BollettinoIncendi_784.pdf
7,2026-09-25,2026-09-25,FG_02,MODERATO,3,20260925_106_BollettinoIncendi_784.pdf
8,2026-09-25,2026-09-25,FG_03,MEDIO,2,20260925_106_BollettinoIncendi_784.pdf
9,2026-09-25,2026-09-25,FG_04,MEDIO,2,20260925_106_BollettinoIncendi_784.pdf


In [ ]:
# Vista riassuntiva: zone × giorni per l'ultimo bollettino
ultimo = df[df["data_emissione"] == df["data_emissione"].max()]
ultimo.pivot(index="zona", columns="data_previsione", values="livello").reindex(ZONE)

data_previsione,2026-09-25,2026-09-26,2026-09-27
zona,,,
BAT,MEDIO,MEDIO,MEDIO
BA_01,MEDIO,MEDIO,BASSO
BA_02,MEDIO,MEDIO,MEDIO
BA_03,MEDIO,MODERATO,MEDIO
BR_01,MODERATO,MEDIO,MEDIO
BR_02,MEDIO,MEDIO,MEDIO
FG_01,MODERATO,MODERATO,MEDIO
FG_02,MODERATO,MODERATO,MEDIO
FG_03,MEDIO,MEDIO,MEDIO
